Installations

In [1]:
pip install pandas scikit-learn matplotlib seaborn pyodbc


  Obtaining dependency information for scikit-learn from https://files.pythonhosted.org/packages/9f/c4/0ab22726a04ede56f689476b760f98f8f46607caecff993017ac1b64aa5d/scikit_learn-1.8.0-cp312-cp312-win_amd64.whl.metadata
  Obtaining dependency information for pyodbc from https://files.pythonhosted.org/packages/c0/70/5e61b216cc13c7f833ef87f4cdeab253a7873f8709253f5076e9bb16c1b3/pyodbc-5.3.0-cp312-cp312-win_amd64.whl.metadata
  Obtaining dependency information for joblib>=1.3.0 from https://files.pythonhosted.org/packages/7b/91/984aca2ec129e2757d1e4e3c81c3fcda9d0f85b74670a094cc443d9ee949/joblib-1.5.3-py3-none-any.whl.metadata
  Obtaining dependency information for threadpoolctl>=3.2.0 from https://files.pythonhosted.org/packages/32/d5/f9a850d79b0851d1d4ef6456097579a9005b31fea68726a4ae5f2d82ddd9/threadpoolctl-3.6.0-py3-none-any.whl.metadata
   ---------------------------------------- 0.0/8.0 MB ? eta -:--:--
   ---------------------------------------- 0.1/8.0 MB 2.0 MB/s eta 0:00:05
   - ---

ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
pandas-profiling 3.2.0 requires joblib~=1.1.0, but you have joblib 1.5.3 which is incompatible.

[notice] A new release of pip is available: 23.2.1 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


Create a new file

In [2]:
import pandas as pd

df = pd.read_csv(r"C:\Users\laksh\OneDrive\Desktop\Project Cleanup\Employee Attrition Analytics\Data\Raw.csv")

print(df.shape)
print(df.head())
print(df['Attrition'].value_counts())

(1470, 35)
   Age Attrition     BusinessTravel  DailyRate              Department  \
0   41       Yes      Travel_Rarely       1102                   Sales   
1   49        No  Travel_Frequently        279  Research & Development   
2   37       Yes      Travel_Rarely       1373  Research & Development   
3   33        No  Travel_Frequently       1392  Research & Development   
4   27        No      Travel_Rarely        591  Research & Development   

   DistanceFromHome  Education EducationField  EmployeeCount  EmployeeNumber  \
0                 1          2  Life Sciences              1               1   
1                 8          1  Life Sciences              1               2   
2                 2          2          Other              1               4   
3                 3          4  Life Sciences              1               5   
4                 2          1        Medical              1               7   

   ...  RelationshipSatisfaction StandardHours  StockOptionLeve

Encode categorical variables

In [3]:
# Convert target variable
df['Attrition'] = df['Attrition'].map({'Yes': 1, 'No': 0})

# Drop columns that add no value
df = df.drop(columns=['EmployeeCount', 'EmployeeNumber', 'Over18', 'StandardHours'])

# Encode categoricals
df = pd.get_dummies(df, drop_first=True)

print(df.shape)
print(df.dtypes)

(1470, 45)
Age                                  int64
Attrition                            int64
DailyRate                            int64
DistanceFromHome                     int64
Education                            int64
EnvironmentSatisfaction              int64
HourlyRate                           int64
JobInvolvement                       int64
JobLevel                             int64
JobSatisfaction                      int64
MonthlyIncome                        int64
MonthlyRate                          int64
NumCompaniesWorked                   int64
PercentSalaryHike                    int64
PerformanceRating                    int64
RelationshipSatisfaction             int64
StockOptionLevel                     int64
TotalWorkingYears                    int64
TrainingTimesLastYear                int64
WorkLifeBalance                      int64
YearsAtCompany                       int64
YearsInCurrentRole                   int64
YearsSinceLastPromotion              int64


Split Data

In [5]:
from sklearn.model_selection import train_test_split

X = df.drop(columns=['Attrition'])
y = df['Attrition']

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

print(X_train.shape, X_test.shape)

(1176, 44) (294, 44)


Train the model

In [7]:
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler

# Scale features
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

# Retrain
model = LogisticRegression(max_iter=2000, random_state=42)
model.fit(X_train_scaled, y_train)

print("Model trained successfully")

Model trained successfully


Model Evaluation

In [8]:
from sklearn.metrics import classification_report, roc_auc_score, confusion_matrix

y_pred = model.predict(X_test_scaled)
y_prob = model.predict_proba(X_test_scaled)[:, 1]

print(classification_report(y_test, y_pred))
print("ROC-AUC Score:", round(roc_auc_score(y_test, y_prob), 2))
print("Confusion Matrix:\n", confusion_matrix(y_test, y_pred))

              precision    recall  f1-score   support

           0       0.92      0.95      0.93       255
           1       0.55      0.44      0.49        39

    accuracy                           0.88       294
   macro avg       0.73      0.69      0.71       294
weighted avg       0.87      0.88      0.87       294

ROC-AUC Score: 0.79
Confusion Matrix:
 [[241  14]
 [ 22  17]]


Top features driving attrition

In [9]:
import pandas as pd

feature_importance = pd.DataFrame({
    'Feature': X.columns,
    'Coefficient': model.coef_[0]
}).sort_values('Coefficient', ascending=False)

print("Top 10 Attrition Drivers:")
print(feature_importance.head(10))

print("\nTop 10 Retention Factors:")
print(feature_importance.tail(10))

Top 10 Attrition Drivers:
                             Feature  Coefficient
43                      OverTime_Yes     0.975324
34     JobRole_Laboratory Technician     0.780808
23  BusinessTravel_Travel_Frequently     0.717950
19                    YearsAtCompany     0.668289
42              MaritalStatus_Single     0.625524
11                NumCompaniesWorked     0.509736
40      JobRole_Sales Representative     0.506542
21           YearsSinceLastPromotion     0.500860
24      BusinessTravel_Travel_Rarely     0.441246
38        JobRole_Research Scientist     0.398999

Top 10 Retention Factors:
                         Feature  Coefficient
18               WorkLifeBalance    -0.255519
29        EducationField_Medical    -0.256401
37     JobRole_Research Director    -0.273562
27  EducationField_Life Sciences    -0.291783
6                 JobInvolvement    -0.339734
4        EnvironmentSatisfaction    -0.412645
8                JobSatisfaction    -0.433434
16             TotalWorkingYe

Export predictions for Tableau

In [11]:
import pandas as pd

results = X_test.copy()
results['Actual_Attrition'] = y_test.values
results['Predicted_Attrition'] = y_pred
results['Attrition_Probability'] = y_prob

results.to_csv(r"C:\Users\laksh\OneDrive\Desktop\Project Cleanup\Employee Attrition Analytics\Data_attrition_predictions.csv", index=False)
print("Exported successfully")

Exported successfully
